# Day 42：TensorBoard 与受控实验

目标：在同一划分、epoch 上限、优化器和种子下，只改变卷积宽度。Day 41 的结果是先修示例，Day 42 的模型选择使用验证集；若反复研究同一测试集，应另留最终评估集。

运行前请阅读[环境与运行说明](../docs/setup.md)。本课 `.py` 是教学源文件，配套 Markdown 和 Notebook 自动同步。图形保存到 `outputs/`，设置 `COURSE_SHOW_PLOTS=1` 可显示窗口。


[Python 源文件](Day%2042.py) · [Notebook](Day%2042.ipynb) · [完整课程目录](../docs/curriculum.md)


In [ ]:
from pathlib import Path
import sys

# 脚本从文件位置定位仓库；Notebook 从当前工作目录向上查找。
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
for candidate in (base, *base.parents):
    if (candidate / "Code" / "course_utils.py").is_file():
        code_dir = str(candidate / "Code")
        if code_dir not in sys.path:
            sys.path.insert(0, code_dir)
        break
else:
    raise FileNotFoundError("找不到课程仓库，请从仓库根目录或 Code 目录启动 Notebook。")
from course_utils import DATA, OUTPUT, finish_plot


## 准备独立实验目录

使用 UTC 时间和唯一后缀避免日志混合，保存清单副本及参数。运行后在仓库根目录执行 `tensorboard --logdir outputs/tensorboard`，浏览器打开 http://localhost:6006。


In [ ]:
import os
import json
from datetime import datetime, timezone
from uuid import uuid4
import tensorflow as tf
from deep_utils import configure, load_manifest, pet_dataset
configure()
manifest = load_manifest()
run_root = OUTPUT / "tensorboard" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
run_root.mkdir(parents=True)
(run_root / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
epochs = 1 if os.environ.get("COURSE_SMOKE") == "1" else 5
results = []


## 只改变宽度

每次重置随机种子、重新创建训练 Dataset，固定数据顺序以尽量保持可比；宽度不同意味着参数形状不同，并非逐个权重相同。早停规则与最大 epoch 相同，实际停止轮数可以不同。TensorBoard 记录训练曲线；EarlyStopping 和 ModelCheckpoint 按验证损失保留最佳状态。硬件仍可能引入微小非确定性。


In [ ]:
def build_model(width):
    return tf.keras.Sequential([
        tf.keras.Input(shape=(64, 64, 3)),
        tf.keras.layers.Conv2D(width, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(width * 2, 3, activation="relu"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])


results = []  # 重跑本单元时清空上次比较结果
for width in [8, 16]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)
    model = build_model(width)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    directory = run_root / f"width_{width}"
    directory.mkdir(exist_ok=True)
    config = {"width": width, "epochs_max": epochs, "seed": 42, "optimizer": "adam",
              "parameters": model.count_params(), "tensorflow": tf.__version__, "keras": tf.keras.__version__}
    (directory / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    model.fit(pet_dataset(manifest, "train", training=True),
              validation_data=pet_dataset(manifest, "validation"), epochs=epochs, verbose=2,
              callbacks=[tf.keras.callbacks.TensorBoard(log_dir=str(directory / "logs")),
                         tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
                         tf.keras.callbacks.ModelCheckpoint(directory / "best.keras", save_best_only=True)])
    # 用实际保存的检查点评估，确保选模指标与后面加载的模型完全对应。
    restored = tf.keras.models.load_model(directory / "best.keras")
    validation = restored.evaluate(pet_dataset(manifest, "validation"), verbose=0, return_dict=True)
    results.append({"width": width, "validation": validation, "path": str(directory / "best.keras")})
print("Validation comparison:", results)


## 选定后只评价一次测试集

验证集成绩用于选择宽度；测试集成绩只报告选定模型，不能用于重新选择。保存结果，记录这只是一次小规模对照，不把更复杂模型默认视为更优。


In [ ]:
best = min(results, key=lambda result: result["validation"]["loss"])
selected = tf.keras.models.load_model(best["path"])
test_metrics = selected.evaluate(pet_dataset(manifest, "test"), verbose=0, return_dict=True)
report = {"experiments": results, "selected_width": best["width"], "test": test_metrics}
(run_root / "results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
print("TensorBoard logs:", run_root)


## 练习与检查

读取 loss 与 val_loss，解释准确率不变但损失变大的情况。进一步实验每次只改一个因素，并记录新的假设；不要靠反复看测试集筛选方案。
